In [ ]:
"""
You can run either this notebook locally (if you have all the dependencies and a GPU) or on Google Colab.

Instructions for setting up Colab are as follows:
1. Open a new Python 3 notebook.
2. Import this notebook from GitHub (File -> Upload Notebook -> "GitHub" tab -> copy/paste GitHub URL)
3. Connect to an instance with a GPU (Runtime -> Change runtime type -> select "GPU" for hardware accelerator)
4. Run this cell to set up dependencies.
5. Restart the runtime (Runtime -> Restart Runtime) for any upgraded packages to take effect


NOTE: User is responsible for checking the content of datasets and the applicable licenses and determining if they are suitable for the intended use.
"""
# If you're using Google Colab and not running locally, run this cell to install dependencies

# Install dependencies
!pip install wget
!apt-get update && apt-get install -y sox libsndfile1 ffmpeg
!pip install text-unidecode
!pip install omegaconf

BRANCH='main'
!python -m pip install git+https://github.com/NVIDIA/NeMo.git@{BRANCH}#egg=nemo_toolkit[asr]

# Introduction

This tutorial introduces the **ASR Inference Pipeline API** — a unified, streaming-first API for speech recognition that lives under `nemo.collections.asr.inference`. It provides a consistent interface for running inference with different ASR models (buffered CTC/RNNT/TDT/SALM, cache-aware CTC/RNNT) and optional post-processing features (e.g. ITN, Translation, ...).


By the end of this tutorial you will know how to:
- Understand the core building blocks: **Frame**, **Stream**, **Pipeline**, **TranscribeStepOutput**
- Build and run different ASR pipelines
- Enable advanced features: **End-of-Utterance (EoU)**, **Word Timestamps**, , **Translation**, **Per-Stream Options** and more

# Dataset Preparation

We use **Mini LibriSpeech** (`dev_clean_2`) — a small subset of LibriSpeech that is also used in other NeMo ASR tutorials. The download script creates a NeMo manifest JSON file that lists every audio file path together with its transcript and duration.

In [ ]:
!mkdir -p datasets/mini-librispeech
!python ../../scripts/dataset_processing/get_librispeech_data.py \
  --data_root datasets/mini-librispeech/ \
  --data_sets dev_clean_2

In [2]:
import json

MANIFEST_PATH = "datasets/mini-librispeech/dev_clean_2.json"
LIMIT = 5

audio_filepaths = []
reference_texts  = []
with open(MANIFEST_PATH) as f:
    for line in f:
        item = json.loads(line)
        audio_filepaths.append(item["audio_filepath"])
        reference_texts.append(item["text"])
        if len(audio_filepaths) == LIMIT:
            break

print(f"Loaded {len(audio_filepaths)} audio files")
for path, ref in zip(audio_filepaths, reference_texts):
    print(f"  {path.split('/')[-1]}  →  {ref}")

Loaded 5 audio files
  1993-147964-0000.wav  →  they sat about the house most of the day as if it were sunday greasing their boots mending their suspenders plaiting whiplashes
  1993-147964-0001.wav  →  anyway he would never allow one of his horses to be put to such a strain
  1993-147964-0002.wav  →  i had wanted to get some picture books for yulka and antonia even yulka was able to read a little now
  1993-147964-0003.wav  →  she cut squares of cotton cloth and we sewed them together into a book
  1993-147964-0004.wav  →  on the white pages i grouped sunday school cards and advertising cards which i had brought from my old country


# Core Concepts

The inference pipeline API is built around the following concepts:

1. **Frame**: An immutable container representing a single chunk of audio.
2. **Stream**: An iterator over an audio source that yields `Frame` objects.
3. **Pipeline**: Responsible for the main inference logic. All pipelines implement the `transcribe_step()` method, which takes a batch of `Frame` objects and processes them. Each call to `transcribe_step()` returns a batch of `TranscribeStepOutput` objects containing all necessary information produced during that step.

## Frame

A `Frame` is a frozen dataclass that wraps a short slice of raw audio samples together with metadata.

Key fields:
- `samples` — 1-D Tensor of audio samples
- `stream_id` — integer identifier for the audio stream this frame belongs to
- `is_first`/`is_last` — boundary flags indicating the first and last chunk of the stream
- `length` — number of *valid* samples in the tensor (the rest may be padded for the last chunk). -1 means the whole tensor is valid

In [3]:
import torch
from nemo.collections.asr.inference.streaming.framing.request import Frame

# Simulate a 160-ms chunk at 16 kHz
SAMPLE_RATE = 16_000
FRAME_SIZE_SECS = 0.16
n_samples = int(SAMPLE_RATE * FRAME_SIZE_SECS)  # 2560 samples

frame = Frame(
    samples=torch.rand(n_samples),
    stream_id=0,
    is_first=True,
    is_last=False,
    length=n_samples,      # all samples are valid (no padding)
)

print(f"stream_id  : {frame.stream_id}")
print(f"is_first   : {frame.is_first}")
print(f"is_last    : {frame.is_last}")
print(f"length     : {frame.length}")

/home/dkaramyan/workspace/niva/sw-ai-app-design/ca/NeMo/env/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/dkaramyan/workspace/niva/sw-ai-app-design/ca/NeMo/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
fused_indices_to_multihot has reached end of life. Please migrate to a non-experimental function.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no tel

stream_id  : 0
is_first   : True
is_last    : False
length     : 2560


## Stream

A `Stream` is an iterator that reads an audio source and emits `Frame` objects. The concrete implementation is provided by the `MonoStream` class, which reads a mono WAV file and slices it into fixed-size chunks.

Key fields:
- `rate` — target sample rate; the file is resampled if necessary.
- `frame_size_in_secs` — duration of each frame in seconds.
- `stream_id` — unique identifier for this stream.
- `pad_last_frame` — if `True`, the last (potentially shorter) frame is zero-padded to `frame_size`.

**Notes:**
- Frames do not overlap each other.
- All frames produced have the same size, except the last frame, which may be shorter.

In [4]:
from nemo.collections.asr.inference.streaming.framing.mono_stream import MonoStream

FRAME_SIZE_SECS = 1.0

stream = MonoStream(
    rate=SAMPLE_RATE,
    frame_size_in_secs=FRAME_SIZE_SECS,
    stream_id=0,
    pad_last_frame=True,
)
stream.load_audio(audio_filepaths[0])

# Iterate through all frames to observe size, valid_size, and boundary flags
print(f"Audio file : {audio_filepaths[0]}")
print(f"Frame size : {FRAME_SIZE_SECS}s ({int(SAMPLE_RATE * FRAME_SIZE_SECS)} samples)")
print()

total_frames = 0
for frames_list in stream:
    frame = frames_list[0]   # MonoStream yields a one-element list
    total_frames += 1
    tag = "FIRST" if frame.is_first else ("LAST" if frame.is_last else "")
    print(f"  Frame {total_frames:1d}  size={frame.size:<5d}  valid_size={frame.valid_size:<5d}  {tag}")
print(f"\nTotal frames: {total_frames}")

Audio file : /home/dkaramyan/workspace/niva/sw-ai-app-design/ca/NeMo/tutorials/asr/datasets/mini-librispeech/LibriSpeech/dev-clean-2-processed/1993-147964-0000.wav
Frame size : 1.0s (16000 samples)

  Frame 1  size=16000  valid_size=16000  FIRST
  Frame 2  size=16000  valid_size=16000  
  Frame 3  size=16000  valid_size=16000  
  Frame 4  size=16000  valid_size=16000  
  Frame 5  size=16000  valid_size=16000  
  Frame 6  size=16000  valid_size=16000  
  Frame 7  size=16000  valid_size=16000  
  Frame 8  size=16000  valid_size=16000  
  Frame 9  size=16000  valid_size=6720   LAST

Total frames: 9


## Multiple Streams

When you need to transcribe **more than one audio** source at the same time, we provide a higher-level wrapper: `MultiStream`. It iterates through all streams until every stream is exhausted.

Key fields:
   - `n_frames_per_stream` — number of frames to use from each stream per iteration step

In [5]:
from nemo.collections.asr.inference.streaming.framing.multi_stream import MultiStream

multi_streamer = MultiStream(n_frames_per_stream=1)
for audio_id, filepath in enumerate(audio_filepaths):
    stream = MonoStream(
        rate=SAMPLE_RATE,
        frame_size_in_secs=FRAME_SIZE_SECS,
        stream_id=audio_id,
        pad_last_frame=True,
    )
    stream.load_audio(filepath)
    multi_streamer.add_stream(stream=stream, stream_id=stream.stream_id)

print(f"Active streams: {len(multi_streamer)}")
print(f"{'Step':<6} {'Batch':>5}  Stream_ids")
print("-" * 40)

frame_counts = {}
for step, batch in enumerate(multi_streamer):
    # batch is a list of Frame objects.
    # shorter files finish earlier, so the batch size shrinks as streams exhaust.
    for frame in batch:
        frame_counts[frame.stream_id] = frame_counts.get(frame.stream_id, 0) + 1

    ids = [f.stream_id for f in batch]
    print(f"{step:<6} {len(batch):>5}  {ids}")

print(f"\nPer-stream frame counts:")
for sid in sorted(frame_counts):
    print(f"  stream {sid}: {frame_counts[sid]:>3} frames")

Active streams: 5
Step   Batch  Stream_ids
----------------------------------------
0          5  [0, 1, 2, 3, 4]
1          5  [0, 1, 2, 3, 4]
2          5  [0, 1, 2, 3, 4]
3          5  [0, 1, 2, 3, 4]
4          5  [0, 1, 2, 3, 4]
5          3  [0, 2, 4]
6          3  [0, 2, 4]
7          2  [0, 4]
8          1  [0]

Per-stream frame counts:
  stream 0:   9 frames
  stream 1:   5 frames
  stream 2:   7 frames
  stream 3:   5 frames
  stream 4:   8 frames


## Continuous Batching

With `MultiStream`, streams are all loaded upfront and the batch shrinks as shorter files finish. **Continuous Batching** keeps the batch full throughout: as soon as one stream ends a new one is automatically started, maximising GPU utilisation.

The `ContinuousBatchedFrameStreamer` class implements this pattern. It wraps `MultiStream` internally and refills it to capacity at every step. Key fields:
- `sample_rate` — target sample rate for all streams
- `frame_size_in_secs` — duration of each frame in seconds
- `batch_size` — maximum number of **concurrently active** streams; new streams are added automatically to maintain this limit
- `n_frames_per_stream` — frames pulled from each active stream per iteration step
- `pad_last_frame` — whether to zero-pad the final (potentially shorter) frame

In [6]:
from nemo.collections.asr.inference.streaming.framing.multi_stream import ContinuousBatchedFrameStreamer
from nemo.collections.asr.inference.streaming.framing.request_options import ASRRequestOptions

SAMPLE_RATE = 16_000
FRAME_SIZE_SECS = 1.0
BATCH_SIZE = 3   # at most 3 streams active simultaneously

streamer = ContinuousBatchedFrameStreamer(
    sample_rate=SAMPLE_RATE,
    frame_size_in_secs=FRAME_SIZE_SECS,
    batch_size=BATCH_SIZE,
    n_frames_per_stream=1,
    pad_last_frame=True,
)

# define per-stream options
options = [ASRRequestOptions() for _ in audio_filepaths]
streamer.set_audio_filepaths(audio_filepaths, options)

print(f"Total files  : {len(audio_filepaths)}")
print(f"Batch size   : {BATCH_SIZE}  (active streams)")
print()
print(f"{'Step':<6} {'Batch':>5}  Stream_ids")
print("-" * 40)

frame_counts = {}
for step, batch in enumerate(streamer):
    # batch shrinks only when fewer than batch_size files remain overall;
    # otherwise it stays at batch_size even as individual streams finish.
    for frame in batch:
        frame_counts[frame.stream_id] = frame_counts.get(frame.stream_id, 0) + 1

    ids = [f.stream_id for f in batch]
    print(f"{step:<6} {len(batch):>5}  {ids}")

print(f"\nPer-stream frame counts:")
for sid in sorted(frame_counts):
    print(f"  stream {sid}: {frame_counts[sid]:>3} frames")

Total files  : 5
Batch size   : 3  (active streams)

Step   Batch  Stream_ids
----------------------------------------
0          3  [0, 1, 2]
1          3  [0, 1, 2]
2          3  [0, 1, 2]
3          3  [0, 1, 2]
4          3  [0, 1, 2]
5          3  [0, 2, 3]
6          3  [0, 2, 3]
7          3  [0, 3, 4]
8          3  [0, 3, 4]
9          2  [3, 4]
10         1  [4]
11         1  [4]
12         1  [4]
13         1  [4]
14         1  [4]

Per-stream frame counts:
  stream 0:   9 frames
  stream 1:   5 frames
  stream 2:   7 frames
  stream 3:   5 frames
  stream 4:   8 frames


## Pipelines

A `Pipeline` is the central inference engine. It wraps the ASR model, feature extraction, buffer/state/cache management, and optional text post-processing (ITN, PnC, translation) into a single object:

> `transcribe_step()` — processes a batch of `Frame` objects and immediately returns one `TranscribeStepOutput` per frame.

`TranscribeStepOutput` carries the following fields:
> - `stream_id` — integer identifying which stream this output belongs to
> - `final_transcript` — text finalized at the **last EoU boundary**; empty on most steps, populated when EoU is detected
> - `partial_transcript` — accumulated text since the **previous EoU**; updated every step (*may change as future frames arrive*)
> - `current_step_transcript` — text obtained from *current* frame only
> - `final_segments` — list of `TextSegment` objects (word/segment metadata) for `final_transcript`

NeMo provides `PipelineBuilder.build_pipeline(cfg)` as a factory that reads an OmegaConf config and builds the corresponding pipeline.

In [7]:
from IPython.display import display, HTML
from collections import defaultdict
from time import sleep

from omegaconf import OmegaConf
from nemo.collections.asr.inference.factory.pipeline_builder import PipelineBuilder, BasePipeline
from nemo.collections.asr.metrics.wer import word_error_rate
from nemo.collections.asr.parts.utils.eval_utils import remove_punctuations


_PALETTE = ["#4a9eff", "#ff7043", "#26a69a", "#ab47bc", "#66bb6a", "#ffa726", "#ef5350", "#42a5f5"]


def create_pipeline(config_path: str, cfg_overrides: dict=None):
    cfg = OmegaConf.load(config_path)
    if cfg_overrides:
        for key, value in cfg_overrides.items():
            OmegaConf.update(cfg, key, value)
    return PipelineBuilder.build_pipeline(cfg)


def compute_wer(hypotheses: list[str], references: list[str]):
    """WER with punctuation and capitalisation ignored (mirrors pipeline_eval defaults)."""
    hyps = [remove_punctuations(h).lower() for h in hypotheses]
    refs = [remove_punctuations(r).lower() for r in references]
    return word_error_rate(hypotheses=hyps, references=refs)


def do_streaming(pipeline: BasePipeline, audio_filepaths: list[str], reference_texts: list[str], wait: float = 0.0):
    sep = pipeline.get_sep()
    # Each pipeline manages its own ContinuousBatchedFrameStreamer internally; get_request_generator() returns it.
    request_generator = pipeline.get_request_generator()
    options = [ASRRequestOptions() for _ in audio_filepaths]
    request_generator.set_audio_filepaths(audio_filepaths, options)

    COLORS = [_PALETTE[i % len(_PALETTE)] for i in range(len(audio_filepaths))]

    accumulated = defaultdict(str)
    cur_partial  = defaultdict(str)
    pipeline_name = type(pipeline).__name__

    def render_html():
        parts = []
        for i in range(len(audio_filepaths)):
            filename = audio_filepaths[i].split("/")[-1]
            color    = COLORS[i]
            final    = accumulated[i]
            partial  = cur_partial[i]
            parts.append(
                f'<div style="margin:4px 0;padding:10px 14px;border-left:4px solid {color};'
                f'background:#fafafa;border-radius:0 4px 4px 0;font-family:serif;">'
                f'<div style="font-size:0.75em;color:#888;margin-bottom:4px;font-family:monospace;">'
                f'stream {i} &nbsp;·&nbsp; {pipeline_name} &nbsp;·&nbsp; {filename}</div>'
                f'<div>{final}{partial}</div>'
                f'</div>'
            )
        return "".join(parts)

    handle = display(HTML(render_html()), display_id=True)

    pipeline.open_session()
    for requests in request_generator:
        step_outputs = pipeline.transcribe_step(requests)
        for out in step_outputs:
            sid = out.stream_id
            if out.final_transcript:
                text = out.final_transcript if accumulated[sid] else out.final_transcript.lstrip(sep)
                accumulated[sid] += text
            cur_partial[sid] = out.partial_transcript
        handle.update(HTML(render_html()))
        if wait > 0:
            sleep(wait)
    pipeline.close_session()

    handle.update(HTML(render_html()))
    transcriptions = dict(accumulated)
    wer = compute_wer(
        [transcriptions[i] for i in range(len(audio_filepaths))],
        reference_texts,
    )
    print(f"WER for {pipeline_name} is: {wer:.2%}")
    return transcriptions

### Buffered CTC/RNNT/TDT Pipeline

<img src="./images/buffered_pipeline.png" alt="Buffered Streaming Pipeline" style="max-width:100%;margin:12px 0;" />

**How it works:** The buffered pipeline adapts any standard offline ASR model (CTC, RNNT, TDT) for streaming without modifying the model architecture. At each step, the pipeline assembles a three-part padded buffer and runs a full forward pass through the model:

- **Left context** — audio from previous steps, providing stable past context so the model "remembers" what came before.
- **Middle chunk** — the newly arrived audio; the primary segment whose tokens will be emitted.
- **Right context** (lookahead) — future audio already received, letting the model resolve ambiguities at the right edge of the middle chunk.

After each forward pass, the pipeline scans the output token sequence from the end of the buffer looking for **N consecutive blank tokens (EoU)**. When EoU is found, all tokens from the start of the middle chunk up to the EoU point are kept.

> **Maximum theoretical latency** = **chunk_size** + **right_padding_size**. The pipeline must wait for the full chunk to arrive and then buffer an additional `right_padding_size` seconds of lookahead before it can emit tokens for that chunk.

In [8]:
batch_size = 8
log_level = 40

# Amount of past audio prepended as left context so the model can condition on what came before the current chunk
left_padding_size = 1.6

# Duration of the middle chunk. Also, equal to frame size
chunk_size = 0.54

# Amount of future (lookahead) audio appended as right context, helping resolve ambiguities at the right edge of the chunk.
right_padding_size = 1.6

# (RNNT only) - When True, the RNNT decoder state is carried over from the previous step, improving transcript continuity across chunks;
#               when False, the decoder state is reset at each step (stateless decoding).
stateful = True

buffered_rnnt_pipeline = create_pipeline(
    "../../examples/asr/conf/asr_streaming_inference/buffered_rnnt.yaml",
    cfg_overrides={
        "asr.model_name": "nvidia/parakeet-rnnt-1.1b",
        "streaming.left_padding_size": left_padding_size,
        "streaming.chunk_size": chunk_size,
        "streaming.right_padding_size": right_padding_size,
        "streaming.batch_size": batch_size,
        "streaming.stateful": stateful,
        "asr_decoding_type": "rnnt",
        "log_level": log_level,
    },
)

buffered_ctc_pipeline = create_pipeline(
    "../../examples/asr/conf/asr_streaming_inference/buffered_ctc.yaml",
    cfg_overrides={
        "asr.model_name": "nvidia/parakeet-ctc-1.1b",
        "streaming.left_padding_size": left_padding_size,
        "streaming.chunk_size": chunk_size,
        "streaming.right_padding_size": right_padding_size,
        "streaming.batch_size": batch_size,
        "asr_decoding_type": "ctc",
        "log_level": log_level,
    },
)

In [9]:
for pipeline in [buffered_rnnt_pipeline, buffered_ctc_pipeline]:
    do_streaming(pipeline, audio_filepaths, reference_texts)

WER for BufferedRNNTPipeline is: 3.23%


WER for BufferedCTCPipeline is: 1.08%


### Cache-Aware CTC/RNNT Pipeline

<img src="./images/cache_aware_pipeline.png" alt="Cache-Aware Streaming Pipeline" style="max-width:100%;margin:12px 0;"/>

**How it works:** Unlike buffered inference — which slides an overlapping window and re-encodes the same audio frames repeatedly — cache-aware streaming processes **each audio frame once**. The FastConformer encoder maintains a **bounded cache of intermediate representations** across all layers (both self-attention KV states and convolutional states). When a new chunk arrives, only that chunk is encoded; past context is read from the cache rather than recomputed. This eliminates redundant activations and enables stable, predictable memory usage regardless of utterance length.

> The latency is determined by the attention context configuration. The format is `[left_context, right_context]`, and the chunk size is equal to `right_context + 1`:
> - `[70, 0]`: Chunk size = 1 (1 * 80ms = 0.08s)  
> - `[70, 1]`: Chunk size = 2 (2 * 80ms = 0.16s)  
> - `[70, 6]`: Chunk size = 7 (7 * 80ms = 0.56s)  
> - `[70, 13]`: Chunk size = 14 (14 * 80ms = 1.12s)

In [10]:
batch_size = 8
log_level = 40

# Maximum number of concurrent stream slots in the cache manager. Must be ≥ batch_size. Controls how many slots should be pre-allocated.
num_slots = 64

# Number of frames the self-attention layers can attend to on the left (past) and right (lookahead) sides. 
# Smaller right context lowers latency.
att_context_size = [70, 13]

ca_rnnt_pipeline = create_pipeline(
    "../../examples/asr/conf/asr_streaming_inference/cache_aware_rnnt.yaml",
    cfg_overrides={
        "asr.model_name": "nvidia/nemotron-speech-streaming-en-0.6b",
        "streaming.batch_size": batch_size,
        "streaming.num_slots": num_slots,
        "streaming.att_context_size": att_context_size,
        "log_level": log_level,
        "asr_decoding_type": "rnnt",
        "enable_pnc": True # nemotron-speech-streaming-en-0.6b supports PnC, so let's enable it
    },
)

ca_ctc_pipeline = create_pipeline(
    "../../examples/asr/conf/asr_streaming_inference/cache_aware_ctc.yaml",
    cfg_overrides={
        "asr.model_name": "stt_en_fastconformer_hybrid_large_streaming_multi",
        "streaming.batch_size": batch_size,
        "streaming.num_slots": num_slots,
        "streaming.att_context_size": att_context_size,
        "asr_decoding_type": "ctc",
        "log_level": log_level,
    },
)

In [11]:
for pipeline in [ca_rnnt_pipeline, ca_ctc_pipeline]:
    do_streaming(pipeline, audio_filepaths, reference_texts)

WER for CacheAwareRNNTPipeline is: 1.08%


WER for CacheAwareCTCPipeline is: 3.23%


In [12]:
# let's remove created pipelines to free up the memory
del buffered_rnnt_pipeline, buffered_ctc_pipeline, ca_rnnt_pipeline, ca_ctc_pipeline

# Advanced Features

We will work with a single audio file throughout rest of the tutorial

In [50]:
! wget https://dldata-public.s3.us-east-2.amazonaws.com/2086-149220-0033.wav

--2026-03-18 20:07:41--  https://dldata-public.s3.us-east-2.amazonaws.com/2086-149220-0033.wav
Resolving dldata-public.s3.us-east-2.amazonaws.com (dldata-public.s3.us-east-2.amazonaws.com)... 3.5.130.229, 52.219.93.130, 3.5.92.118, ...
Connecting to dldata-public.s3.us-east-2.amazonaws.com (dldata-public.s3.us-east-2.amazonaws.com)|3.5.130.229|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 237964 (232K) [audio/wav]
Saving to: ‘2086-149220-0033.wav’

2086-149220-0033.wa 100%[===================>] 232.39K   406KB/s    in 0.6s    

2026-03-18 20:07:43 (406 KB/s) - ‘2086-149220-0033.wav’ saved [237964/237964]



In [ ]:
audio_filepath = "2086-149220-0033.wav"

## EoU Detection

EoU detection enables the pipeline to automatically **segment a continuous audio stream into discrete, finalized utterances** — without needing explicit stream boundaries from the caller.

**How it works:** After each forward pass the pipeline inspects the token sequence in the buffer. When a long-enough run of **silent (blank) tokens** is observed at the trailing edge of the buffer, it concludes that the speaker has paused and marks an EoU boundary. At that moment:
- `final_transcript` is populated with the text accumulated since the **previous** EoU boundary.
- `partial_transcript` is reset to an empty string.

***Note**: Different pipelines implement different EoU detection logic.*

In [58]:
pipeline = create_pipeline(
    "../../examples/asr/conf/asr_streaming_inference/buffered_rnnt.yaml",
    cfg_overrides={
        "asr.model_name": "nvidia/parakeet-rnnt-1.1b",
        "streaming.left_padding_size": 1.6,
        "streaming.chunk_size": 0.54,
        "streaming.right_padding_size": 1.6,
        "streaming.batch_size": 8,
        "streaming.stateful": True,
        "asr_decoding_type": "rnnt",
        # ---- EoU parameters ----
        "endpointing.stop_history_eou": 400,     # silence window in ms (400ms corresponds to 5 blank token)
        "endpointing.residue_tokens_at_end": 2,  # tail tokens excluded from scan
        "log_level": 40,
    },
)

In [75]:
def stream_with_eou(pipeline, audio_filepath):
    """
    Stream a single audio file through *pipeline* and print each EoU event
    together with the partial transcript updates along the way.
    """
    request_generator = pipeline.get_request_generator()
    request_generator.set_audio_filepaths([audio_filepath], [ASRRequestOptions()])

    sep = pipeline.get_sep()
    accumulated_final = ""
    text_to_show = ""

    pipeline.open_session()
    for step, requests in enumerate(request_generator):
        step_outputs = pipeline.transcribe_step(requests)
        for out in step_outputs:
            if out.final_transcript:
                # Strip leading separator on the very first segment
                text = out.final_transcript if accumulated_final else out.final_transcript.lstrip(sep)
                accumulated_final += text + "[EoU-🎬]"
            partial_transcript = out.partial_transcript
            text_to_show = f"{accumulated_final}{partial_transcript}".strip()
            print(f"Step#{step:<3} -> {text_to_show}")
    pipeline.close_session()
    print("-" * 100)
    final_text = accumulated_final.replace("[EoU-🎬]", "")
    print(f"Full transcription: {final_text}")

stream_with_eou(pipeline, audio_filepath)

Step#0   -> 
Step#1   -> 
Step#2   -> 
Step#3   -> well
Step#4   -> well i don't w
Step#5   -> well i don't wish to see it
Step#6   -> well i don't wish to see it any more ob
Step#7   -> well i don't wish to see it any more observed phoe
Step#8   -> well i don't wish to see it any more observed phoebe turning away her eyes[EoU-🎬]
Step#9   -> well i don't wish to see it any more observed phoebe turning away her eyes[EoU-🎬]
Step#10  -> well i don't wish to see it any more observed phoebe turning away her eyes[EoU-🎬]
Step#11  -> well i don't wish to see it any more observed phoebe turning away her eyes[EoU-🎬] it
Step#12  -> well i don't wish to see it any more observed phoebe turning away her eyes[EoU-🎬] it is certain
Step#13  -> well i don't wish to see it any more observed phoebe turning away her eyes[EoU-🎬] it is certainly very like the old portrait[EoU-🎬]
----------------------------------------------------------------------------------------------------
Full transcription: well i don

## Word Timestamps

## Translation

## Per-Stream Options

# What's Next?

# Limitations

>**Feature availability:**
Not all features are supported by all pipeline types.
>
>**Standalone scripts:**
There are standalone inference scripts for buffered and cache-aware streaming under [`examples/asr/asr_chunked_inference/`](../../examples/asr/asr_chunked_inference/) and [`examples/asr/asr_cache_aware_streaming/`](../../examples/asr/asr_cache_aware_streaming/). Performance may vary between those scripts because the Pipeline API feeds fixed-size buffers into the model. 